In [ ]:
!pip install -q fastapi uvicorn websockets pyngrok openai faster-whisper nest_asyncio numpy
!apt-get install -q ffmpeg

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

import json
import numpy as np
import threading
import time
from contextlib import asynccontextmanager

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request
import uvicorn
from pyngrok import ngrok, conf
import openai
from faster_whisper import WhisperModel
import torch
import concurrent.futures
import os

# ── CONFIG ────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
_secrets       = UserSecretsClient()
OPENAI_API_KEY = _secrets.get_secret("OPENAI_API_KEY")
NGROK_TOKEN    = _secrets.get_secret("NGROK_TOKEN")

PORT = 8000
conf.get_default().auth_token = NGROK_TOKEN
openai_client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)

# ── DEVICE ────────────────────────────────────────────────────────
CPU_CORES = os.cpu_count()
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE   = "float16" if DEVICE == "cuda" else "int8"
print(f"Device: {DEVICE}  |  CPU cores: {CPU_CORES}")

# ── LOAD SILERO VAD ───────────────────────────────────────────────
print("Loading Silero VAD …")
vad_model, _ = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    trust_repo=True,
)
vad_model.eval()
vad_model = vad_model.cpu()
print("Silero VAD loaded ✓")

# ── LOAD WHISPER ──────────────────────────────────────────────────
print("Loading Whisper large-v3 …")
whisper_model = WhisperModel(
    "large-v3",
    device=DEVICE,
    compute_type=COMPUTE,
    cpu_threads=CPU_CORES,
    num_workers=2,
)
print(f"Whisper loaded ✓  (device={DEVICE}, compute={COMPUTE})")

_executor = concurrent.futures.ThreadPoolExecutor(
    max_workers=4,
    thread_name_prefix="whisper",
)

# ── VAD CONSTANTS ─────────────────────────────────────────────────
VAD_SAMPLE_RATE    = 16000
VAD_FRAME_SIZE     = 512
VAD_CHUNK_SECS     = 0.1
VAD_SPEECH_THRESH  = 0.5
SILENCE_FIRE_SECS  = 0.8    # pause this long → fire transcription & keep listening
SILENCE_RESET_SECS = 2.0    # pause this long → full reset (new utterance)
MIN_SPEECH_SECS    = 0.3    # ignore sub-300ms blips
# ── TRANSLATION PROMPTS ───────────────────────────────────────────
PROMPT_EN_UR = (
    "You are a professional real-time translation engine. "
    "Keep technical terms in English — do not translate words like "
    "architecture, computer, encoder, transformer, object, class, "
    "inheritance, encapsulation, polymorphism, etc. "
    "Translate the English input to Urdu. "
    "Output ONLY the Urdu translation — no explanations, no notes."
)

PROMPT_UR_EN = (
    "You are a professional real-time translation engine. "
    "Translate the Urdu input to English. "
    "Output ONLY the English translation — no explanations, no notes."
)

# ── APP STATE ─────────────────────────────────────────────────────
connections  = {"teacher": None, "student": None}
floor_holder = "teacher"
floor_lock   = asyncio.Lock()
public_url   = ""


async def broadcast(msg: dict):
    for ws in connections.values():
        if ws:
            try:
                await ws.send_text(json.dumps(msg))
            except Exception:
                pass


async def notify(role: str, msg: dict):
    ws = connections.get(role)
    if ws:
        try:
            await ws.send_text(json.dumps(msg))
        except Exception:
            pass


# ── VAD HELPER ────────────────────────────────────────────────────
# Silero requires exactly VAD_FRAME_SIZE (512) samples per call.
# We iterate over the buffer in 512-sample frames and return the
# maximum probability across all frames — this way any frame with
# speech counts as speech for the whole chunk.
def _vad_speech_prob(audio_np: np.ndarray) -> float:
    audio_f32 = audio_np.astype(np.float32) / 32768.0
    max_prob   = 0.0
    n          = len(audio_f32)

    for start in range(0, n, VAD_FRAME_SIZE):
        frame = audio_f32[start : start + VAD_FRAME_SIZE]

        # Pad the last frame if it's shorter than VAD_FRAME_SIZE
        if len(frame) < VAD_FRAME_SIZE:
            frame = np.pad(frame, (0, VAD_FRAME_SIZE - len(frame)))

        tensor = torch.from_numpy(frame).cpu()
        with torch.no_grad():
            prob = vad_model(tensor, VAD_SAMPLE_RATE).item()
        if prob > max_prob:
            max_prob = prob

    return max_prob


# ── LIVE TRANSLATION SESSION ──────────────────────────────────────
class LiveTranslationSession:

    def __init__(self, source, target, tts_voice, prompt, whisper_lang):
        self.source       = source
        self.target       = target
        self.tts_voice    = tts_voice
        self.prompt       = prompt
        self.whisper_lang = whisper_lang
        self.direction    = f"{source}→{target}"

        self.mic_queue       = asyncio.Queue()
        self.translate_queue = asyncio.Queue()

        self._buf_lock = asyncio.Lock()
        self._buf      = []

        self._drainer_task  = None
        self._vad_task      = None
        self._pipeline_task = None
        self._active_tts    = None

    def start(self):
        self._drainer_task  = asyncio.create_task(self._drainer_loop())
        self._vad_task      = asyncio.create_task(self._vad_loop())
        self._pipeline_task = asyncio.create_task(self._pipeline_loop())
        print(f"  [{self.direction}] session ready ✓  (lang={self.whisper_lang})")

    async def put_audio(self, pcm: bytes):
        await self.mic_queue.put(pcm)

    async def drain(self):
        if self._active_tts and not self._active_tts.done():
            self._active_tts.cancel()
            try:
                await self._active_tts
            except asyncio.CancelledError:
                pass
        self._active_tts = None
        for q in (self.mic_queue, self.translate_queue):
            while not q.empty():
                try:    q.get_nowait()
                except asyncio.QueueEmpty: break
        async with self._buf_lock:
            self._buf.clear()

    # ── DRAINER: mic_queue → raw buffer ──────────────────────────
    async def _drainer_loop(self):
        while True:
            pcm   = await self.mic_queue.get()
            chunk = np.frombuffer(pcm, dtype=np.int16)
            async with self._buf_lock:
                self._buf.append(chunk)

    async def _vad_loop(self):
        await broadcast({"type": "session_ready", "direction": self.direction})
    
        speech_buf      = []
        silence_counter = 0.0
        speech_detected = False
        fired           = False   # have we fired for this utterance already?
    
        while True:
            await asyncio.sleep(VAD_CHUNK_SECS)
    
            async with self._buf_lock:
                if not self._buf:
                    chunk = None
                else:
                    chunk     = np.concatenate(self._buf)
                    self._buf = []
    
            if chunk is None:
                if speech_detected:
                    silence_counter += VAD_CHUNK_SECS
            else:
                try:
                    speech_prob = _vad_speech_prob(chunk)
                except Exception as e:
                    print(f"  [{self.direction}] VAD error: {e!r}")
                    continue
    
                if speech_prob >= VAD_SPEECH_THRESH:
                    speech_buf.append(chunk)
                    silence_counter = 0.0
                    speech_detected = True
                    fired = False          # new speech resets the fired flag
                    buf_secs = sum(len(c) for c in speech_buf) / VAD_SAMPLE_RATE
                    print(
                        f"  [{self.direction}] speech "
                        f"prob={speech_prob:.2f}  buf={buf_secs:.1f}s"
                    )
                else:
                    if speech_detected:
                        speech_buf.append(chunk)   # keep for context
                        silence_counter += VAD_CHUNK_SECS
    
            total_secs = sum(len(c) for c in speech_buf) / VAD_SAMPLE_RATE
            has_enough = total_secs >= MIN_SPEECH_SECS
    
            # ── FIRE: silence long enough and we haven't fired yet ──
            if (speech_detected and has_enough
                    and silence_counter >= SILENCE_FIRE_SECS
                    and not fired):
                print(
                    f"  [{self.direction}] firing transcription "
                    f"({total_secs:.1f}s, pause={silence_counter:.1f}s)"
                )
                audio_np = np.concatenate(speech_buf)
                fired    = True
                asyncio.create_task(self._transcribe_and_enqueue(audio_np))
                # DO NOT reset speech_buf — keep accumulating for next fire
    
            # ── RESET: long silence = truly done speaking ───────────
            if speech_detected and silence_counter >= SILENCE_RESET_SECS:
                print(f"  [{self.direction}] reset — long silence, new utterance")
                speech_buf      = []
                silence_counter = 0.0
                speech_detected = False
                fired           = False
        # ── TRANSCRIBE ────────────────────────────────────────────────
        async def _transcribe_and_enqueue(self, audio_np: np.ndarray):
            audio_f32 = audio_np.astype(np.float32) / 32768.0
            duration  = len(audio_f32) / VAD_SAMPLE_RATE
            print(f"  [{self.direction}] transcribing {duration:.1f}s …")
    
            loop = asyncio.get_event_loop()
            try:
                segments, _ = await loop.run_in_executor(
                    _executor,
                    lambda: whisper_model.transcribe(
                        audio_f32,
                        language=self.whisper_lang,
                        beam_size=5,
                        vad_filter=False,                  # Silero already gated this
                        condition_on_previous_text=False,  # prevents hallucination loops
                    )
                )
                transcript = " ".join(s.text for s in segments).strip()
            except Exception as e:
                print(f"  [{self.direction}] whisper error: {e!r}")
                return
    
            if not transcript:
                print(f"  [{self.direction}] empty transcript, skipping")
                return
    
            print(f"  [{self.direction}] transcript: '{transcript}'")
            await self.translate_queue.put(transcript)

    # ── PIPELINE LOOP ─────────────────────────────────────────────
    async def _pipeline_loop(self):
        while True:
            text = await self.translate_queue.get()
            self._active_tts = asyncio.create_task(
                self._translate_and_speak(text)
            )
            try:
                await self._active_tts
            except asyncio.CancelledError:
                pass
            finally:
                self._active_tts = None

    # ── TRANSLATE + TTS ───────────────────────────────────────────
    async def _translate_and_speak(self, text: str):
        try:
            resp = await openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": self.prompt},
                    {"role": "user",   "content": text},
                ],
                temperature=0.2,
                max_tokens=600,
            )
            translated = resp.choices[0].message.content.strip()
            if not translated:
                return

            print(f"  [{self.direction}] translated: '{translated}'")

            target_ws = connections.get(self.target)
            if not target_ws:
                print(f"  [{self.direction}] target not connected, dropping TTS")
                return

            n = 0
            async with openai_client.audio.speech.with_streaming_response.create(
                model="gpt-4o-mini-tts",
                voice=self.tts_voice,
                input=translated,
                response_format="pcm",
            ) as tts_resp:
                async for chunk in tts_resp.iter_bytes(chunk_size=8192):
                    try:
                        await target_ws.send_bytes(chunk)
                        n += 1
                    except Exception as e:
                        print(f"  [{self.direction}] send_bytes failed: {e!r}")
                        return

            print(f"  [{self.direction}] →{self.target}: sent {n} TTS chunks ✓")

        except asyncio.CancelledError:
            print(f"  [{self.direction}] TTS cancelled (floor change)")
            raise
        except Exception as e:
            print(f"  [{self.direction}] pipeline error: {e!r}")


# ── SESSIONS ──────────────────────────────────────────────────────
session_t2s = LiveTranslationSession(
    source="teacher", target="student",
    tts_voice="shimmer",
    prompt=PROMPT_EN_UR,
    whisper_lang="en",
)

session_s2t = LiveTranslationSession(
    source="student", target="teacher",
    tts_voice="shimmer",
    prompt=PROMPT_UR_EN,
    whisper_lang="ur",
)

# ── HTML PAGES ────────────────────────────────────────────────────
TEACHER_PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Teacher — Classroom Translator</title>
  <link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">
  <style>
    :root {
      --bg:#0d1117;--surface:#161b22;--surface2:#1c2128;
      --border:#30363d;--teacher:#58a6ff;--teacher-glow:rgba(88,166,255,0.18);
      --student:#3fb950;--warn:#d29922;--danger:#f85149;
      --text:#e6edf3;--muted:#7d8590;--radius:14px;
    }
    *{box-sizing:border-box;margin:0;padding:0}
    body{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);
      min-height:100vh;display:flex;flex-direction:column;align-items:center;
      justify-content:center;padding:24px;
      background-image:radial-gradient(ellipse at 20% 20%,rgba(88,166,255,0.06) 0%,transparent 50%),
                        radial-gradient(ellipse at 80% 80%,rgba(63,185,80,0.04) 0%,transparent 50%);}
    .header{text-align:center;margin-bottom:32px}
    .header h1{font-family:'DM Serif Display',serif;font-size:2rem;letter-spacing:-0.02em}
    .badge{display:inline-flex;align-items:center;gap:6px;
      background:var(--teacher-glow);border:1px solid var(--teacher);color:var(--teacher);
      font-size:0.78rem;font-weight:600;letter-spacing:0.08em;text-transform:uppercase;
      padding:4px 12px;border-radius:100px;margin-top:10px}
    .card{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);
      padding:32px;width:100%;max-width:480px;box-shadow:0 8px 32px rgba(0,0,0,0.4)}
    .floor-display{background:var(--surface2);border:1px solid var(--border);border-radius:10px;
      padding:18px 20px;margin-bottom:24px;display:flex;align-items:center;gap:12px}
    .floor-icon{font-size:1.8rem}
    .floor-label{font-size:0.72rem;color:var(--muted);text-transform:uppercase;letter-spacing:0.08em;margin-bottom:2px}
    .floor-name{font-size:1rem;font-weight:600}
    .floor-name.teacher{color:var(--teacher)}
    .floor-name.student{color:var(--student)}
    #hand-alert{display:none;background:rgba(210,153,34,0.12);border:1px solid var(--warn);
      border-radius:10px;padding:16px 20px;margin-bottom:20px}
    #hand-alert.show{display:flex;align-items:center;gap:12px}
    .alert-text{flex:1;font-size:0.9rem;color:var(--warn);font-weight:500}
    .btn-row{display:flex;gap:10px;margin-bottom:20px}
    button{flex:1;padding:13px 16px;border-radius:10px;border:none;
      font-family:'DM Sans',sans-serif;font-size:0.88rem;font-weight:600;
      cursor:pointer;transition:all 0.18s ease}
    button:disabled{opacity:0.4;cursor:not-allowed}
    #btn-start{background:var(--teacher);color:#0d1117}
    #btn-start:hover:not(:disabled){filter:brightness(1.1);transform:translateY(-1px)}
    #btn-allow{background:var(--student);color:#0d1117}
    #btn-take{background:var(--surface2);border:1px solid var(--border);color:var(--text)}
    #btn-take:hover:not(:disabled){border-color:var(--teacher);color:var(--teacher)}
    .mic-bar{display:flex;align-items:center;gap:10px;padding:12px 16px;
      background:var(--surface2);border-radius:10px;border:1px solid var(--border);margin-bottom:20px}
    .mic-dot{width:10px;height:10px;border-radius:50%;background:var(--muted);transition:background 0.3s}
    .mic-dot.active{background:var(--teacher);animation:blink 1s ease infinite}
    .mic-dot.muted{background:var(--danger)}
    @keyframes blink{0%,100%{opacity:1}50%{opacity:0.3}}
    .mic-label{font-size:0.82rem;color:var(--muted)}
    .mic-label span{color:var(--text);font-weight:500}
    .status-row{display:flex;gap:10px;margin-bottom:20px}
    .status-pill{flex:1;padding:8px 12px;border-radius:8px;background:var(--surface2);
      border:1px solid var(--border);font-size:0.75rem;text-align:center}
    .status-pill .dot{display:inline-block;width:7px;height:7px;border-radius:50%;
      background:var(--muted);margin-right:5px}
    .status-pill.ok .dot{background:var(--student)}
    .status-pill.err .dot{background:var(--danger)}
    #log{background:var(--surface2);border:1px solid var(--border);border-radius:10px;
      padding:12px 14px;font-size:0.73rem;color:var(--muted);max-height:160px;
      overflow-y:auto;font-family:monospace;line-height:1.6}
    #log:empty::before{content:'Activity log will appear here...'}
  </style>
</head>
<body>
<div class="header">
  <h1>🎓 Classroom Translator</h1>
  <div class="badge">👨‍🏫 Teacher Panel</div>
</div>
<div class="card">
  <div class="floor-display">
    <div class="floor-icon" id="floor-icon">🎙️</div>
    <div>
      <div class="floor-label">Current Speaker</div>
      <div class="floor-name teacher" id="floor-name">Teacher (You)</div>
    </div>
  </div>
  <div id="hand-alert">
    <span style="font-size:1.4rem">✋</span>
    <div class="alert-text">Student is raising their hand!</div>
    <button id="btn-allow" onclick="allowStudent()" style="flex:0;padding:8px 16px">Allow</button>
  </div>
  <div class="btn-row">
    <button id="btn-start" onclick="startSession()">▶ Start Session</button>
    <button id="btn-take" onclick="takeFloor()" disabled>🎙️ Take Floor Back</button>
  </div>
  <div class="mic-bar">
    <div class="mic-dot" id="mic-dot"></div>
    <div class="mic-label">Mic: <span id="mic-label">Idle</span></div>
  </div>
  <div class="status-row">
    <div class="status-pill" id="pill-ws"><span class="dot"></span>Server</div>
    <div class="status-pill" id="pill-stt"><span class="dot"></span>Whisper</div>
    <div class="status-pill" id="pill-student"><span class="dot"></span>Student</div>
  </div>
  <div id="log"></div>
</div>
<script>
const WS_URL = "WS_TEACHER_PLACEHOLDER";
let ws, playCtx, nextPlayTime = 0;
let iHaveFloor = true, sessionStarted = false;
let micCtx, micSrc, micProc;

function log(m) {
  const el = document.getElementById('log');
  const ts = new Date().toLocaleTimeString('en-GB',{hour12:false});
  el.innerHTML += '['+ts+'] '+m+'<br>';
  el.scrollTop = el.scrollHeight;
}
function setPill(id,ok){document.getElementById(id).className='status-pill '+(ok?'ok':'err')}

function updateFloorUI(holder) {
  iHaveFloor = (holder === 'teacher');
  document.getElementById('floor-icon').textContent = iHaveFloor ? '🎙️' : '👂';
  const fn = document.getElementById('floor-name');
  fn.textContent = iHaveFloor ? 'Teacher (You)' : 'Student';
  fn.className = 'floor-name '+(iHaveFloor?'teacher':'student');
  document.getElementById('btn-take').disabled = iHaveFloor;
  if (!sessionStarted) return;
  const dot = document.getElementById('mic-dot');
  const lbl = document.getElementById('mic-label');
  if (iHaveFloor) {
    dot.className='mic-dot active'; lbl.textContent='Live — speak in English';
  } else {
    dot.className='mic-dot muted'; lbl.textContent='Listening (student speaking)';
  }
}

async function startSession() {
  document.getElementById('btn-start').disabled = true;
  log('Requesting microphone...');
  try {
    const stream = await navigator.mediaDevices.getUserMedia({
      audio:{sampleRate:16000,channelCount:1,echoCancellation:true,noiseSuppression:true}
    });
    playCtx = new (window.AudioContext||window.webkitAudioContext)({sampleRate:24000});
    if (playCtx.state==='suspended') await playCtx.resume();
    nextPlayTime = playCtx.currentTime;

    micCtx  = new (window.AudioContext||window.webkitAudioContext)({sampleRate:16000});
    micSrc  = micCtx.createMediaStreamSource(stream);
    micProc = micCtx.createScriptProcessor(4096,1,1);
    micProc.onaudioprocess = (e) => {
      if (!ws||ws.readyState!==WebSocket.OPEN||!iHaveFloor) return;
      const inp = e.inputBuffer.getChannelData(0);
      const pcm = new Int16Array(inp.length);
      for (let i=0;i<inp.length;i++)
        pcm[i]=Math.max(-32768,Math.min(32767,Math.round(inp[i]*32768)));
      ws.send(pcm.buffer);
    };
    micSrc.connect(micProc);
    micProc.connect(micCtx.destination);

    ws = new WebSocket(WS_URL);
    ws.binaryType = 'arraybuffer';
    ws.onopen  = ()=>{sessionStarted=true;setPill('pill-ws',true);log('✅ Connected');updateFloorUI('teacher')};
    ws.onclose = ()=>{setPill('pill-ws',false);log('❌ Disconnected')};
    ws.onerror = ()=>{log('❌ WebSocket error')};
    ws.onmessage = (ev)=>{
      if (typeof ev.data==='string'){handleControl(JSON.parse(ev.data));return;}
      playAudio(ev.data);
    };
    log('✅ Session started — live translation active');
  } catch(err) {
    log('❌ '+err.message);
    document.getElementById('btn-start').disabled = false;
  }
}

function handleControl(msg) {
  if (msg.type==='hand_raised')          {document.getElementById('hand-alert').classList.add('show');log('✋ Student raised hand')}
  if (msg.type==='floor_status')         {document.getElementById('hand-alert').classList.remove('show');updateFloorUI(msg.holder);log('🎙️ Floor → '+msg.holder)}
  if (msg.type==='session_ready')        {setPill('pill-stt',true);log('🤖 Whisper ready ('+msg.direction+')')}
  if (msg.type==='student_connected')    {setPill('pill-student',true);log('👨‍🎓 Student connected')}
  if (msg.type==='student_disconnected') {setPill('pill-student',false);log('👨‍🎓 Student left')}
}

function allowStudent(){
  if (!ws||ws.readyState!==WebSocket.OPEN) return;
  ws.send(JSON.stringify({type:'allow_student'}));
  document.getElementById('hand-alert').classList.remove('show');
  log('✅ Student given the floor');
}
function takeFloor(){
  if (!ws||ws.readyState!==WebSocket.OPEN) return;
  ws.send(JSON.stringify({type:'take_floor'}));
  log('🎙️ Reclaiming floor...');
}

function playAudio(data) {
  if (!playCtx) return;
  const int16 = new Int16Array(data);
  const f32 = new Float32Array(int16.length);
  for (let i=0;i<int16.length;i++) f32[i]=int16[i]/32768.0;
  const buf = playCtx.createBuffer(1,f32.length,24000);
  buf.copyToChannel(f32,0);
  const src = playCtx.createBufferSource();
  src.buffer=buf; src.connect(playCtx.destination);
  if (playCtx.state!=='running') playCtx.resume();
  const t = Math.max(playCtx.currentTime+0.05,nextPlayTime);
  src.start(t);
  nextPlayTime = t+buf.duration;
}
</script>
</body>
</html>"""


STUDENT_PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Student — Classroom Translator</title>
  <link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">
  <style>
    :root{--bg:#0d1117;--surface:#161b22;--surface2:#1c2128;--border:#30363d;
      --teacher:#58a6ff;--student:#3fb950;--student-glow:rgba(63,185,80,0.18);
      --warn:#d29922;--danger:#f85149;--text:#e6edf3;--muted:#7d8590;--radius:14px}
    *{box-sizing:border-box;margin:0;padding:0}
    body{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);
      min-height:100vh;display:flex;flex-direction:column;align-items:center;
      justify-content:center;padding:24px;
      background-image:radial-gradient(ellipse at 80% 20%,rgba(63,185,80,0.06) 0%,transparent 50%),
                        radial-gradient(ellipse at 20% 80%,rgba(88,166,255,0.04) 0%,transparent 50%)}
    .header{text-align:center;margin-bottom:32px}
    .header h1{font-family:'DM Serif Display',serif;font-size:2rem;letter-spacing:-0.02em}
    .badge{display:inline-flex;align-items:center;gap:6px;
      background:var(--student-glow);border:1px solid var(--student);color:var(--student);
      font-size:0.78rem;font-weight:600;letter-spacing:0.08em;text-transform:uppercase;
      padding:4px 12px;border-radius:100px;margin-top:10px}
    .card{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);
      padding:32px;width:100%;max-width:480px;box-shadow:0 8px 32px rgba(0,0,0,0.4)}
    .floor-display{background:var(--surface2);border:1px solid var(--border);border-radius:10px;
      padding:18px 20px;margin-bottom:24px;display:flex;align-items:center;gap:12px}
    .floor-icon{font-size:1.8rem}
    .floor-label{font-size:0.72rem;color:var(--muted);text-transform:uppercase;letter-spacing:0.08em;margin-bottom:2px}
    .floor-name{font-size:1rem;font-weight:600}
    .floor-name.teacher{color:var(--teacher)}
    .floor-name.student{color:var(--student)}
    #hand-status{display:none;border-radius:10px;padding:14px 18px;margin-bottom:20px;
      font-size:0.88rem;font-weight:500;text-align:center}
    #hand-status.waiting{display:block;background:rgba(210,153,34,0.1);border:1px solid var(--warn);color:var(--warn)}
    #hand-status.granted{display:block;background:rgba(63,185,80,0.1);border:1px solid var(--student);color:var(--student)}
    .btn-row{display:flex;gap:10px;margin-bottom:20px}
    button{flex:1;padding:13px 16px;border-radius:10px;border:none;
      font-family:'DM Sans',sans-serif;font-size:0.88rem;font-weight:600;
      cursor:pointer;transition:all 0.18s ease}
    button:disabled{opacity:0.4;cursor:not-allowed}
    #btn-start{background:var(--student);color:#0d1117}
    #btn-start:hover:not(:disabled){filter:brightness(1.1);transform:translateY(-1px)}
    #btn-hand{background:var(--surface2);border:1px solid var(--border);color:var(--text)}
    #btn-hand:hover:not(:disabled){border-color:var(--warn);color:var(--warn)}
    #btn-hand.raised{border-color:var(--warn);color:var(--warn);background:rgba(210,153,34,0.08);animation:pulse 1.4s ease infinite}
    @keyframes pulse{0%,100%{box-shadow:0 0 0 0 rgba(210,153,34,0.4)}50%{box-shadow:0 0 0 8px rgba(210,153,34,0)}}
    .mic-bar{display:flex;align-items:center;gap:10px;padding:12px 16px;
      background:var(--surface2);border-radius:10px;border:1px solid var(--border);margin-bottom:20px}
    .mic-dot{width:10px;height:10px;border-radius:50%;background:var(--muted);transition:background 0.3s}
    .mic-dot.active{background:var(--student);animation:blink 1s ease infinite}
    .mic-dot.muted{background:var(--danger)}
    @keyframes blink{0%,100%{opacity:1}50%{opacity:0.3}}
    .mic-label{font-size:0.82rem;color:var(--muted)}
    .mic-label span{color:var(--text);font-weight:500}
    .status-row{display:flex;gap:10px;margin-bottom:20px}
    .status-pill{flex:1;padding:8px 12px;border-radius:8px;background:var(--surface2);
      border:1px solid var(--border);font-size:0.75rem;text-align:center}
    .status-pill .dot{display:inline-block;width:7px;height:7px;border-radius:50%;
      background:var(--muted);margin-right:5px}
    .status-pill.ok .dot{background:var(--student)}
    .status-pill.err .dot{background:var(--danger)}
    #log{background:var(--surface2);border:1px solid var(--border);border-radius:10px;
      padding:12px 14px;font-size:0.73rem;color:var(--muted);max-height:160px;
      overflow-y:auto;font-family:monospace;line-height:1.6}
    #log:empty::before{content:'Activity log will appear here...'}
  </style>
</head>
<body>
<div class="header">
  <h1>🎓 Classroom Translator</h1>
  <div class="badge">👨‍🎓 Student Panel</div>
</div>
<div class="card">
  <div class="floor-display">
    <div class="floor-icon" id="floor-icon">👂</div>
    <div>
      <div class="floor-label">Current Speaker</div>
      <div class="floor-name teacher" id="floor-name">Teacher</div>
    </div>
  </div>
  <div id="hand-status"></div>
  <div class="btn-row">
    <button id="btn-start" onclick="startSession()">▶ Join Session</button>
    <button id="btn-hand" onclick="raiseHand()" disabled>✋ Raise Hand</button>
  </div>
  <div class="mic-bar">
    <div class="mic-dot" id="mic-dot"></div>
    <div class="mic-label">Mic: <span id="mic-label">Idle</span></div>
  </div>
  <div class="status-row">
    <div class="status-pill" id="pill-ws"><span class="dot"></span>Server</div>
    <div class="status-pill" id="pill-stt"><span class="dot"></span>Whisper</div>
    <div class="status-pill" id="pill-teacher"><span class="dot"></span>Teacher</div>
  </div>
  <div id="log"></div>
</div>
<script>
const WS_URL = "WS_STUDENT_PLACEHOLDER";
let ws, playCtx, nextPlayTime = 0;
let iHaveFloor = false, sessionStarted = false, handRaised = false;

function log(m){
  const el=document.getElementById('log');
  const ts=new Date().toLocaleTimeString('en-GB',{hour12:false});
  el.innerHTML+='['+ts+'] '+m+'<br>';
  el.scrollTop=el.scrollHeight;
}
function setPill(id,ok){document.getElementById(id).className='status-pill '+(ok?'ok':'err')}

function updateFloorUI(holder) {
  iHaveFloor = (holder==='student');
  document.getElementById('floor-icon').textContent = iHaveFloor ? '🎙️' : '👂';
  const fn = document.getElementById('floor-name');
  fn.textContent = iHaveFloor ? 'Student (You)' : 'Teacher';
  fn.className = 'floor-name '+(iHaveFloor?'student':'teacher');
  const hs = document.getElementById('hand-status');
  if (iHaveFloor) {
    hs.textContent='✅ You have the floor — speak in Urdu';
    hs.className='hand-status granted'; hs.style.display='block';
    document.getElementById('btn-hand').disabled=true;
    document.getElementById('btn-hand').className='';
    handRaised=false;
  } else {
    hs.style.display='none'; hs.className='';
    if (sessionStarted) document.getElementById('btn-hand').disabled=false;
  }
  if (!sessionStarted) return;
  const dot=document.getElementById('mic-dot');
  const lbl=document.getElementById('mic-label');
  if (iHaveFloor){
    dot.className='mic-dot active'; lbl.textContent='Live — Urdu میں بولیں';
  } else {
    dot.className='mic-dot muted'; lbl.textContent='Listening (teacher speaking)';
  }
}

async function startSession() {
  document.getElementById('btn-start').disabled = true;
  log('Requesting microphone...');
  try {
    const stream = await navigator.mediaDevices.getUserMedia({
      audio:{sampleRate:16000,channelCount:1,echoCancellation:true,noiseSuppression:true}
    });
    playCtx = new (window.AudioContext||window.webkitAudioContext)({sampleRate:24000});
    if (playCtx.state==='suspended') await playCtx.resume();
    nextPlayTime = playCtx.currentTime;

    const micCtx  = new (window.AudioContext||window.webkitAudioContext)({sampleRate:16000});
    const micSrc  = micCtx.createMediaStreamSource(stream);
    const micProc = micCtx.createScriptProcessor(4096,1,1);
    micProc.onaudioprocess = (e)=>{
      if (!ws||ws.readyState!==WebSocket.OPEN||!iHaveFloor) return;
      const inp=e.inputBuffer.getChannelData(0);
      const pcm=new Int16Array(inp.length);
      for (let i=0;i<inp.length;i++)
        pcm[i]=Math.max(-32768,Math.min(32767,Math.round(inp[i]*32768)));
      ws.send(pcm.buffer);
    };
    micSrc.connect(micProc);
    micProc.connect(micCtx.destination);

    ws = new WebSocket(WS_URL);
    ws.binaryType = 'arraybuffer';
    ws.onopen  = ()=>{
      sessionStarted=true;
      setPill('pill-ws',true);
      document.getElementById('btn-hand').disabled=false;
      log('✅ Connected — live translation active');
    };
    ws.onclose = ()=>{setPill('pill-ws',false);log('❌ Disconnected')};
    ws.onerror = ()=>{log('❌ WebSocket error')};
    ws.onmessage = (ev)=>{
      if (typeof ev.data==='string'){handleControl(JSON.parse(ev.data));return;}
      playAudio(ev.data);
    };
    log('✅ Joined session');
  } catch(err) {
    log('❌ '+err.message);
    document.getElementById('btn-start').disabled=false;
  }
}

function handleControl(msg) {
  if (msg.type==='floor_status')         {updateFloorUI(msg.holder);log('🎙️ Floor → '+msg.holder)}
  if (msg.type==='session_ready')        {setPill('pill-stt',true);log('🤖 Whisper ready ('+msg.direction+')')}
  if (msg.type==='teacher_connected')    {setPill('pill-teacher',true);log('👨‍🏫 Teacher connected')}
  if (msg.type==='teacher_disconnected') {setPill('pill-teacher',false);log('👨‍🏫 Teacher left')}
}

function raiseHand(){
  if (!ws||ws.readyState!==WebSocket.OPEN||handRaised) return;
  handRaised=true;
  ws.send(JSON.stringify({type:'raise_hand'}));
  const hs=document.getElementById('hand-status');
  hs.textContent='✋ Hand raised — waiting for teacher...';
  hs.className='hand-status waiting'; hs.style.display='block';
  document.getElementById('btn-hand').className='raised';
  document.getElementById('btn-hand').disabled=true;
  log('✋ Hand raised');
}

function playAudio(data) {
  if (!playCtx) return;
  const int16=new Int16Array(data);
  const f32=new Float32Array(int16.length);
  for (let i=0;i<int16.length;i++) f32[i]=int16[i]/32768.0;
  const buf=playCtx.createBuffer(1,f32.length,24000);
  buf.copyToChannel(f32,0);
  const src=playCtx.createBufferSource();
  src.buffer=buf; src.connect(playCtx.destination);
  if (playCtx.state!=='running') playCtx.resume();
  const t=Math.max(playCtx.currentTime+0.05,nextPlayTime);
  src.start(t);
  nextPlayTime=t+buf.duration;
}
</script>
</body>
</html>"""


# ── FASTAPI ───────────────────────────────────────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    session_t2s.start()
    session_s2t.start()
    await asyncio.sleep(1)
    yield

app = FastAPI(lifespan=lifespan)

class NgrokHeaderMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        response.headers["ngrok-skip-browser-warning"] = "true"
        return response

app.add_middleware(NgrokHeaderMiddleware)


@app.get("/teacher", response_class=HTMLResponse)
async def teacher_page():
    ws_url = (public_url
              .replace("https://", "wss://")
              .replace("http://",  "ws://") + "/ws/teacher")
    return TEACHER_PAGE.replace("WS_TEACHER_PLACEHOLDER", ws_url)


@app.get("/student", response_class=HTMLResponse)
async def student_page():
    ws_url = (public_url
              .replace("https://", "wss://")
              .replace("http://",  "ws://") + "/ws/student")
    return STUDENT_PAGE.replace("WS_STUDENT_PLACEHOLDER", ws_url)


@app.websocket("/ws/{role}")
async def ws_handler(websocket: WebSocket, role: str):
    global floor_holder

    if role not in ("teacher", "student"):
        await websocket.close(code=1008)
        return

    await websocket.accept()
    connections[role] = websocket
    other = "student" if role == "teacher" else "teacher"
    print(f"\n  [{role.upper()}] connected")

    await notify(other, {"type": f"{role}_connected"})
    await websocket.send_text(
        json.dumps({"type": "floor_status", "holder": floor_holder})
    )

    async def receive_loop():
        global floor_holder
        try:
            while True:
                msg = await websocket.receive()
                if msg.get("type") == "websocket.disconnect":
                    break

                if msg.get("bytes"):
                    async with floor_lock:
                        holder = floor_holder
                    if role != holder:
                        continue
                    if role == "teacher":
                        await session_t2s.put_audio(msg["bytes"])
                    else:
                        await session_s2t.put_audio(msg["bytes"])

                elif msg.get("text"):
                    ctrl  = json.loads(msg["text"])
                    ctype = ctrl.get("type")
                    print(f"  [{role}] ctrl: {ctype}")

                    if ctype == "raise_hand" and role == "student":
                        await notify("teacher", {"type": "hand_raised"})

                    elif ctype == "allow_student" and role == "teacher":
                        async with floor_lock:
                            floor_holder = "student"
                        await session_t2s.drain()
                        await broadcast({"type": "floor_status", "holder": "student"})
                        print("  Floor → student")

                    elif ctype == "take_floor" and role == "teacher":
                        async with floor_lock:
                            floor_holder = "teacher"
                        await session_s2t.drain()
                        await broadcast({"type": "floor_status", "holder": "teacher"})
                        print("  Floor → teacher")

        except WebSocketDisconnect:
            print(f"  [{role}] disconnected")
        except RuntimeError as e:
            if "disconnect" in str(e).lower():
                print(f"  [{role}] disconnected")
            else:
                print(f"  [{role}] runtime error: {e!r}")
        except Exception as e:
            print(f"  [{role}] error: {e!r}")

    await receive_loop()
    connections[role] = None
    await notify(other, {"type": f"{role}_disconnected"})
    print(f"  [{role.upper()}] cleaned up")


# ── LAUNCH (KAGGLE) ───────────────────────────────────────────────
tunnel     = ngrok.connect(PORT)
public_url = tunnel.public_url

print("\n" + "="*60)
print("  CLASSROOM TRANSLATOR  —  Silero VAD edition")
print("="*60)
print(f"  TEACHER  →  {public_url}/teacher")
print(f"  STUDENT  →  {public_url}/student")
print("="*60)
print(f"  STT      : faster-whisper large-v3  ({DEVICE})")
print(f"  VAD      : Silero — 512-sample frames, silence-gated")
print(f"  Silence  : {SILENCE_FIRE_SECS}s pause triggers transcription")
print(f"  Reset    : {SILENCE_RESET_SECS}s silence resets utterance")
print()


def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    cfg    = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
    server = uvicorn.Server(cfg)
    loop.run_until_complete(server.serve())


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

time.sleep(2)
print("  Server running ✓")
print()

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n  Shutting down …")
    try:
        ngrok.disconnect(public_url)
    except Exception:
        pass